# Colab-ready 教學入口：Ch14 深入 TensorFlow 機制

本 notebook 由官方程式碼 notebook 產生，第一格加入 Colab setup，讓學生不需要手動 clone repo 或切換工作目錄。

- 官方來源：`ch14/ch14_part3.ipynb`
- 官方 repo：https://github.com/rasbt/python-machine-learning-book-3rd-edition.git
- 書籍：Sebastian Raschka and Vahid Mirjalili, *Python Machine Learning, 3rd Ed.*, Packt Publishing, 2019
- 程式碼授權：MIT License，請參考本 repo 的 `THIRD_PARTY_NOTICES.md`

上課時請先執行下一格 setup，再依序執行原 notebook。若深度學習或大型資料章節耗時過久，請改用課堂 quick mode 或由講師示範重點 cell。


In [ ]:
# @title Colab setup for Python Machine Learning 3rd ed.
import importlib
import os
import platform
import subprocess
import sys

REPO_URL = "https://github.com/rasbt/python-machine-learning-book-3rd-edition.git"
REPO_DIR = "/content/python-machine-learning-book-3rd-edition" if os.path.exists("/content") else os.path.abspath("_python_ml_3e_official")
CHAPTER_DIR = "ch14"
EXTRA_PACKAGES = ['watermark', 'mlxtend', 'pyprind', 'tensorflow', 'tensorflow-datasets']

def _run(cmd):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)

if not os.path.isdir(REPO_DIR):
    _run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])

target_dir = os.path.join(REPO_DIR, CHAPTER_DIR)
os.chdir(target_dir)
print("Working directory:", os.getcwd())

def ensure_import(import_name, pip_name=None):
    pip_name = pip_name or import_name
    if importlib.util.find_spec(import_name) is None:
        _run([sys.executable, "-m", "pip", "install", "-q", pip_name])

for import_name, pip_name in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("sklearn", "scikit-learn"),
    ("scipy", "scipy"),
]:
    ensure_import(import_name, pip_name)

for pkg in EXTRA_PACKAGES:
    import_name = pkg.split("==")[0].split("[")[0].replace("-", "_")
    if pkg.startswith("tensorflow-datasets"):
        import_name = "tensorflow_datasets"
    if pkg.startswith("scikit-learn"):
        import_name = "sklearn"
    if pkg.startswith("gym=="):
        import_name = "gym"
    ensure_import(import_name, pkg)

import numpy as np

# Compatibility shims for the current Colab runtime family.
# Colab 2026.04 lists Python 3.12.13, NumPy 2.0.2, and TensorFlow 2.19.0.
# The 2019 book notebooks still use a few aliases/API shapes from older releases.
if not hasattr(np, "float"):
    np.float = float
if not hasattr(np, "int"):
    np.int = int
if not hasattr(np, "bool8"):
    np.bool8 = np.bool_

try:
    import matplotlib
    import matplotlib.pyplot as plt
    matplotlib.rcParams["figure.figsize"] = (7, 5)
except Exception as exc:
    print("matplotlib setup skipped:", type(exc).__name__, exc)

try:
    import tensorflow as tf
    print("TensorFlow devices:", [device.device_type + ":" + device.name.split(":")[-1] for device in tf.config.list_physical_devices()])
except Exception as exc:
    print("TensorFlow not loaded:", type(exc).__name__, exc)

try:
    import gym

    if not getattr(gym, "_pyml3e_old_api_patch", False):
        _original_gym_make = gym.make

        class _OldStepAPIWrapper(gym.Wrapper):
            def reset(self, *args, **kwargs):
                result = self.env.reset(*args, **kwargs)
                if isinstance(result, tuple) and len(result) == 2:
                    return result[0]
                return result

            def step(self, action):
                result = self.env.step(action)
                if isinstance(result, tuple) and len(result) == 5:
                    obs, reward, terminated, truncated, info = result
                    return obs, reward, bool(terminated or truncated), info
                return result

        def _patched_make(*args, **kwargs):
            env = _original_gym_make(*args, **kwargs)
            return _OldStepAPIWrapper(env)

        gym.make = _patched_make
        gym._pyml3e_old_api_patch = True
        print("Gym old-step API wrapper enabled.")
except Exception as exc:
    print("Gym compatibility setup skipped:", type(exc).__name__, exc)

print("Python:", sys.version.split()[0], "| Platform:", platform.platform())
for mod_name in ["numpy", "pandas", "matplotlib", "sklearn", "tensorflow", "tensorflow_datasets", "gym"]:
    try:
        mod = importlib.import_module(mod_name)
        print(f"{mod_name}:", getattr(mod, "__version__", "installed"))
    except Exception as exc:
        print(f"{mod_name}: not loaded ({type(exc).__name__})")

print("Setup complete. Run the notebook cells below in order.")


*Python Machine Learning 3rd Edition* by [Sebastian Raschka](https://sebastianraschka.com) & [Vahid Mirjalili](http://vahidmirjalili.com), Packt Publishing Ltd. 2019

Code Repository: https://github.com/rasbt/python-machine-learning-book-3rd-edition

Code License: [MIT License](https://github.com/rasbt/python-machine-learning-book-3rd-edition/blob/master/LICENSE.txt)

# Chapter 14: Going Deeper -- the Mechanics of TensorFlow (Part 3/3)

Note that the optional watermark extension is a small IPython notebook plugin that I developed to make the code reproducible. You can just skip the following line(s).

In [ ]:
%load_ext watermark
%watermark -a "Sebastian Raschka & Vahid Mirjalili" -u -d -p numpy,scipy,tensorflow,tensorflow_datasets


In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np


### Using Estimators for MNIST hand-written digit classification

In [ ]:
BUFFER_SIZE = 10000
BATCH_SIZE = 64
NUM_EPOCHS = 20
steps_per_epoch = np.ceil(60000 / BATCH_SIZE)


In [ ]:
def preprocess(item):
    image = item['image']
    label = item['label']
    image = tf.image.convert_image_dtype(
        image, tf.float32)
    image = tf.reshape(image, (-1,))

    return {'image-pixels':image}, label[..., tf.newaxis]

#Step 1: Defining the input functions (one for training and one for evaluation)
## Step 1: Define the input function for training
def train_input_fn():
    datasets = tfds.load(name='mnist')
    mnist_train = datasets['train']

    dataset = mnist_train.map(preprocess)
    dataset = dataset.shuffle(BUFFER_SIZE)
    dataset = dataset.batch(BATCH_SIZE)
    return dataset.repeat()

## define input-function for evaluation:
def eval_input_fn():
    datasets = tfds.load(name='mnist')
    mnist_test = datasets['test']
    dataset = mnist_test.map(preprocess).batch(BATCH_SIZE)
    return dataset


In [ ]:
## Step 2: feature column
image_feature_column = tf.feature_column.numeric_column(
    key='image-pixels', shape=(28*28))


In [ ]:
## Step 3: instantiate the estimator
dnn_classifier = tf.estimator.DNNClassifier(
    feature_columns=[image_feature_column],
    hidden_units=[32, 16],
    n_classes=10,
    model_dir='models/mnist-dnn/')


## Step 4: train
dnn_classifier.train(
    input_fn=train_input_fn,
    steps=NUM_EPOCHS * steps_per_epoch)


In [ ]:
eval_result = dnn_classifier.evaluate(
    input_fn=eval_input_fn)

print(eval_result)


### Creating a custom Estimator from an existing Keras model

In [ ]:
## Set random seeds for reproducibility
tf.random.set_seed(1)
np.random.seed(1)

## Create the data
x = np.random.uniform(low=-1, high=1, size=(200, 2))
y = np.ones(len(x))
y[x[:, 0] * x[:, 1]<0] = 0

x_train = x[:100, :]
y_train = y[:100]
x_valid = x[100:, :]
y_valid = y[100:]


In [ ]:
## Step 1: Define the input functions
def train_input_fn(x_train, y_train, batch_size=8):
    dataset = tf.data.Dataset.from_tensor_slices(
        ({'input-features':x_train}, y_train.reshape(-1, 1)))

    # Shuffle, repeat, and batch the examples.
    return dataset.shuffle(100).repeat().batch(batch_size)

def eval_input_fn(x_test, y_test=None, batch_size=8):
    if y_test is None:
        dataset = tf.data.Dataset.from_tensor_slices(
            {'input-features':x_test})
    else:
        dataset = tf.data.Dataset.from_tensor_slices(
            ({'input-features':x_test}, y_test.reshape(-1, 1)))


    # Shuffle, repeat, and batch the examples.
    return dataset.batch(batch_size)


In [ ]:
## Step 2: Define the feature columns
features = [
    tf.feature_column.numeric_column(
        key='input-features:', shape=(2,))
]
    
features


In [ ]:
## Step 3: Create the estimator: convert from a Keras model
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(2,), name='input-features'),
    tf.keras.layers.Dense(units=4, activation='relu'),
    tf.keras.layers.Dense(units=4, activation='relu'),
    tf.keras.layers.Dense(units=4, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.summary()

model.compile(optimizer=tf.keras.optimizers.SGD(),
              loss=tf.keras.losses.BinaryCrossentropy(),
              metrics=[tf.keras.metrics.BinaryAccuracy()])

my_estimator = tf.keras.estimator.model_to_estimator(
    keras_model=model,
    model_dir='models/estimator-for-XOR/')


In [ ]:
## Step 4: use the estimator: train/evaluate/predict

num_epochs = 200
batch_size = 2
steps_per_epoch = np.ceil(len(x_train) / batch_size)

my_estimator.train(
    input_fn=lambda: train_input_fn(x_train, y_train, batch_size),
    steps=num_epochs * steps_per_epoch)


In [ ]:
my_estimator.evaluate(
    input_fn=lambda: eval_input_fn(x_valid, y_valid, batch_size))


...

# Summary

...

---

Readers may ignore the next cell.

In [ ]:
! python ../.convert_notebook_to_script.py --input ch14_part3.ipynb --output ch14_part3.py
